In [1]:
import pandas as pd
import random
import numpy as np

In [2]:
FILE = "/workspace/data/gps_cleaned.csv"

# Read ONLY the ID column to save RAM
print("Scanning Trip IDs...")
df_ids = pd.read_csv(FILE, usecols=['trip_id'])

# Get unique IDs and sample 10% of them
all_trips = df_ids['trip_id'].unique()
sampled_trips = set(random.sample(list(all_trips), k=int(len(all_trips) * 0.05)))

print(f"Selected {len(sampled_trips)} trips out of {len(all_trips)}")

# Read the file in chunks and filter
chunk_size = 100000
processed_chunks = []

print("Reading and filtering data...")
for chunk in pd.read_csv(FILE, chunksize=chunk_size):
    # Keep row if its trip_id is in our lucky list
    filtered_chunk = chunk[chunk['trip_id'].isin(sampled_trips)]
    processed_chunks.append(filtered_chunk)

df_final = pd.concat(processed_chunks)

print(f"Final dataset shape: {df_final.shape}")
print(df_final.head())

Scanning Trip IDs...
Selected 170412 trips out of 1704127
Reading and filtering data...
Final dataset shape: (7846842, 5)
                 trip_id   taxi_id   timestamp  longitude   latitude
269  1372637274620000403  20000403  1372637274  -8.611794  41.140557
270  1372637274620000403  20000403  1372637289  -8.611785  41.140575
271  1372637274620000403  20000403  1372637304  -8.612001  41.140566
272  1372637274620000403  20000403  1372637319  -8.612622  41.140503
273  1372637274620000403  20000403  1372637334  -8.613702  41.140341


In [3]:
# Calculate speed

df = df_final.sort_values(by=['trip_id', 'timestamp'])

# Shift each cell for faster calculation (same row)
df['prev_lat'] = df.groupby('trip_id')['latitude'].shift(1)
df['prev_lon'] = df.groupby('trip_id')['longitude'].shift(1)
df['prev_time'] = df.groupby('trip_id')['timestamp'].shift(1)

# Calculate distance between two coord
def haversine_vectorized(lat1, lon1, lat2, lon2):
    R = 6371000  # Radius of Earth in meters
    
    # Convert degrees to radians
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlambda = np.radians(lon2 - lon1)
    
    # Haversine formula
    a = np.sin(dphi/2)**2 + \
        np.cos(phi1) * np.cos(phi2) * np.sin(dlambda/2)**2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
    
    return R * c

df['dist_meters'] = haversine_vectorized(
    df['prev_lat'], df['prev_lon'],
    df['latitude'], df['longitude']
)

df['time_diff'] = df['timestamp'] - df['prev_time']

df['speed_mps'] = df['dist_meters'] / df['time_diff']

df['speed_mps'] = df['speed_mps'].fillna(0)

df.drop(columns=['prev_time', 'dist_meters', 'time_diff'], inplace=True)

print(df[['trip_id', 'timestamp', 'speed_mps']].head(10))

                   trip_id   timestamp  speed_mps
12794  1372637084620000285  1372637084   0.000000
12795  1372637084620000285  1372637099   2.706710
12796  1372637084620000285  1372637114   3.358392
12797  1372637084620000285  1372637129   6.637189
12798  1372637084620000285  1372637144   6.708838
12799  1372637084620000285  1372637159   7.947266
12800  1372637084620000285  1372637174  13.671083
12801  1372637084620000285  1372637189  12.262038
12802  1372637084620000285  1372637204  12.012762
12803  1372637084620000285  1372637219   8.356338


In [4]:
# Calculate bearing (degrees)

def calculate_bearing(lat1, lon1, lat2, lon2):
    lat1, lon1 = np.radians(lat1), np.radians(lon1)
    lat2, lon2 = np.radians(lat2), np.radians(lon2)
    dLon = lon2 - lon1

    x = np.sin(dLon) * np.cos(lat2)
    y = np.cos(lat1) * np.sin(lat2) - (np.sin(lat1) * np.cos(lat2) * np.cos(dLon))
    
    initial_bearing = np.arctan2(x, y)
    
    initial_bearing = np.degrees(initial_bearing)
    compass_bearing = (initial_bearing + 360) % 360
    
    return compass_bearing

df['bearing'] = calculate_bearing(
    df['prev_lat'], df['prev_lon'],
    df['latitude'], df['longitude']
)

df['bearing'] = df['bearing'].fillna(0)
df['bearing_sin'] = np.sin(np.radians(df['bearing']))
df['bearing_cos'] = np.cos(np.radians(df['bearing']))

df.drop(columns=['prev_lat', 'prev_lon'], inplace=True)

print(df[['trip_id', 'timestamp', 'bearing','bearing_sin','bearing_cos','speed_mps']].head(10))

                   trip_id   timestamp     bearing  bearing_sin  bearing_cos  \
12794  1372637084620000285  1372637084    0.000000     0.000000     1.000000   
12795  1372637084620000285  1372637099  350.383957    -0.167045     0.985949   
12796  1372637084620000285  1372637114  260.855635    -0.987291    -0.158923   
12797  1372637084620000285  1372637129  174.352923     0.098401    -0.995147   
12798  1372637084620000285  1372637144  230.472196    -0.771316    -0.636453   
12799  1372637084620000285  1372637159  116.956968     0.891347    -0.453321   
12800  1372637084620000285  1372637174  123.466363     0.834210    -0.551447   
12801  1372637084620000285  1372637189  130.350671     0.762096    -0.647464   
12802  1372637084620000285  1372637204   78.141001     0.978656     0.205504   
12803  1372637084620000285  1372637219   61.376869     0.877790     0.479046   

       speed_mps  
12794   0.000000  
12795   2.706710  
12796   3.358392  
12797   6.637189  
12798   6.708838  
12799

In [5]:
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
import numpy as np

df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.dropna(inplace=True)

feature_cols = ['latitude', 'longitude', 'speed_mps', 'bearing_sin', 'bearing_cos']
target_cols = ['latitude', 'longitude']

scaler = MinMaxScaler()
df[feature_cols] = scaler.fit_transform(df[feature_cols])

# We split trips, not random rows, so the model learns full trajectories
unique_trips = df['trip_id'].unique()
train_trips, test_trips = train_test_split(unique_trips, test_size=0.2, random_state=42)

train_df = df[df['trip_id'].isin(train_trips)]
test_df = df[df['trip_id'].isin(test_trips)]

print(f"Training on {len(train_trips)} trips, Testing on {len(test_trips)} trips.")

Training on 136329 trips, Testing on 34083 trips.


In [6]:
def create_sequences(df, lookback=10):
    X, y = [], []

    # Process each trip separately to avoid jumping between different trips in one sequence
    for _, group in df.groupby('trip_id'):
        data = group[feature_cols].values
        targets = group[target_cols].values

        # If trip is shorter than lookback, skip it
        if len(data) <= lookback:
            continue

        # Create sliding window
        for i in range(lookback, len(data)):
            X.append(data[i - lookback:i])  # Past 'lookback' steps
            y.append(targets[i])  # Current step (prediction target)

    return np.array(X), np.array(y)


LOOKBACK = 10  # How many previous GPS points the model sees
print("Generating sequences... (this may take a moment)")

X_train, y_train = create_sequences(train_df, LOOKBACK)
X_test, y_test = create_sequences(test_df, LOOKBACK)

print(f"X_train shape: {X_train.shape} (Samples, Timesteps, Features)")
print(f"y_train shape: {y_train.shape} (Samples, Targets)")

Generating sequences... (this may take a moment)
X_train shape: (4970311, 10, 5) (Samples, Timesteps, Features)
y_train shape: (4970311, 2) (Samples, Targets)


In [7]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

# Define Model
model = Sequential([
    LSTM(64, return_sequences=True, input_shape=(X_train.shape[1], X_train.shape[2])),
    Dropout(0.2),
    LSTM(32, return_sequences=False),
    Dropout(0.2),
    Dense(2)
])

model.compile(optimizer='adam', loss='mse')
model.summary()

# Train
history = model.fit(
    X_train, y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.1,
    verbose=1
)

2026-01-19 20:18:26.566747: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-01-19 20:18:26.617451: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-01-19 20:18:28.299712: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-01-19 20:18:29.132343: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
/opt/conda/envs/gps-analytics/lib/python3.12/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 10, 64)         │        17,920 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 10, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 2)              │            66 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 30,402 (118.76 KB)

 Trainable params: 30,402 (118.76 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10


2026-01-19 20:18:32.171348: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 894655800 exceeds 10% of free system memory.


21305/69895 ━━━━━━━━━━━━━━━━━━━━ 9:13 11ms/step - loss: 0.0017

KeyboardInterrupt: 

In [ ]:
predictions_scaled = model.predict(X_test)


def inverse_transform_targets(pred_scaled, actual_scaled, scaler):
    # Create placeholders for other features (fill with 0)
    # We only care about columns 0 (lat) and 1 (lon)
    dummy_pred = np.zeros((len(pred_scaled), len(feature_cols)))
    dummy_actual = np.zeros((len(actual_scaled), len(feature_cols)))

    # Fill Lat/Lon columns
    dummy_pred[:, 0:2] = pred_scaled
    dummy_actual[:, 0:2] = actual_scaled

    # Inverse transform
    res_pred = scaler.inverse_transform(dummy_pred)[:, 0:2]
    res_actual = scaler.inverse_transform(dummy_actual)[:, 0:2]

    return res_pred, res_actual


pred_coords, actual_coords = inverse_transform_targets(predictions_scaled, y_test, scaler)


# Calculate Error in Meters (Accuracy Verification)
# We reuse your existing haversine logic, but adapted for arrays
def haversine_np(lat1, lon1, lat2, lon2):
    R = 6371000
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlambda = np.radians(lon2 - lon1)
    a = np.sin(dphi / 2) ** 2 + np.cos(phi1) * np.cos(phi2) * np.sin(dlambda / 2) ** 2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
    return R * c


errors_meters = haversine_np(
    actual_coords[:, 0], actual_coords[:, 1],
    pred_coords[:, 0], pred_coords[:, 1]
)

print(f"--- Accuracy Report ---")
print(f"Mean Error: {np.mean(errors_meters):.2f} meters")
print(f"Median Error: {np.median(errors_meters):.2f} meters")
print(f"Max Error: {np.max(errors_meters):.2f} meters")

results_df = pd.DataFrame({
    'Actual_Lat': actual_coords[:, 0],
    'Actual_Lon': actual_coords[:, 1],
    'Pred_Lat': pred_coords[:, 0],
    'Pred_Lon': pred_coords[:, 1],
    'Error_Meters': errors_meters
})

print("\nSample Predictions:")
print(results_df.sample(5))

In [ ]:
import joblib

model.save('gps_lstm_model.keras')
print("Model saved to gps_lstm_model.keras")

joblib.dump(scaler, 'gps_scaler.pkl')
print("Scaler saved to gps_scaler.pkl")